# 02 — From two rankings to final evidence

**10–15 minute lab.** Follow one query through semantic and lexical rankings, combine them with
Reciprocal Rank Fusion (RRF), and inspect the citable evidence returned to generation.

**Flow:** semantic + BM25 → RRF → evidence.


In [ ]:
from raglab import Citation, ProvenanceStatus
from raglab.retrieval import RankingTrace, RetrievalResult, RetrievedChunk
from raglab.retrieval.ranking import reciprocal_rank_fusion


def make_chunk(chunk_id, content, *, ann_distance=None, bm25_score=None):
    return RetrievedChunk(
        chunk_id=chunk_id,
        document_id="aster-manual",
        chunk_index=int(chunk_id.rsplit("-", 1)[1]),
        content=content,
        token_count=len(content.split()),
        heading_path=("Fault diagnosis",),
        embedding=(1.0, 0.0),
        citation=Citation(
            source_uri="memory://aster-manual",
            source_name="aster-manual.md",
            title="Aster Greenhouse Controller",
            heading_path=("Fault diagnosis",),
            start_page=None,
            end_page=None,
            start_line=20,
            end_line=24,
            provenance_status=ProvenanceStatus.COMPLETE,
        ),
        ann_distance=ann_distance,
        bm25_score=bm25_score,
    )


## Checkpoint 1 — Objective: compare semantic and BM25 rankings

**Run:** inspect controlled results for the query `How do I recover from fault E17?`.


In [ ]:
query = "How do I recover from fault E17?"
semantic = [
    make_chunk("chunk-1", "Low irrigation flow triggers a protective shutdown.", ann_distance=0.08),
    make_chunk(
        "chunk-2",
        "Inspect the flow sensor before restarting irrigation.",
        ann_distance=0.12,
    ),
    make_chunk("chunk-3", "Reset procedure for fault E17.", ann_distance=0.20),
]
lexical = [
    make_chunk("chunk-3", "Reset procedure for fault E17.", bm25_score=8.4),
    make_chunk("chunk-2", "Inspect the flow sensor before restarting irrigation.", bm25_score=4.1),
    make_chunk("chunk-1", "Low irrigation flow triggers a protective shutdown.", bm25_score=2.0),
]

print("QUERY")
print(f"  {query}")
print("\nSEMANTIC RESULTS")
for rank, item in enumerate(semantic, start=1):
    print(
        f"  Rank {rank} | Distance: {item.ann_distance:.2f} | "
        f"Source: {item.citation.source_name}"
    )
    print(f"    Evidence: {item.content}")
print("\nBM25 RESULTS")
for rank, item in enumerate(lexical, start=1):
    print(
        f"  Rank {rank} | BM25 score: {item.bm25_score:.1f} | "
        f"Source: {item.citation.source_name}"
    )
    print(f"    Evidence: {item.content}")
print("\nRANKING EXPLANATION")
print("  Semantic search favors meaning; BM25 favors exact terms such as E17.")
print("  Their raw scores are not comparable, so the next checkpoint fuses ranks.")

### What to observe

Semantic search ranks the paraphrased meaning first; BM25 ranks the exact identifier `E17`
first. Their raw scores use different scales and must not be added directly.

### Conclusion

The two channels are complementary: meaning finds paraphrases, while BM25 protects exact terms.


## Checkpoint 2 — Objective: fuse ranks with RRF

**Run:** call RAGLab's real ranking function. RRF rewards high positions in either list.


In [ ]:
fused = reciprocal_rank_fusion([semantic], [lexical], k=60)
print("QUERY")
print(f"  {query}")
print("\nFUSED RETRIEVAL RESULTS")
for rank, item in enumerate(fused, start=1):
    print(
        f"  Rank {rank} | Semantic rank: {item.ann_rank} | "
        f"BM25 rank: {item.bm25_rank} | RRF score: {item.rrf_score:.5f}"
    )
    print(f"    Source: {item.chunk.citation.source_name}")
    print(f"    Evidence: {item.chunk.content}")
print("\nRANKING EXPLANATION")
print("  RRF rewards high positions in either list; it does not add incompatible raw scores.")

### What to observe

Expect chunks that rank well across both channels to rise. The output keeps each channel's rank,
so the fused order remains explainable.

### Conclusion

RRF combines ordinal evidence, avoiding a false comparison between distance and BM25 score.


## Checkpoint 3 — Objective: return faithful evidence

**Run:** convert the top fused candidates into the public retrieval result contract.


In [ ]:
evidence = tuple(
    RetrievalResult(
        id=f"result-{position}",
        document_id=item.chunk.document_id,
        content=item.chunk.content,
        citation=item.chunk.citation,
        matched_chunk_ids=(item.chunk.chunk_id,),
        first_chunk_index=item.chunk.chunk_index,
        last_chunk_index=item.chunk.chunk_index,
        trace=RankingTrace(
            ann_rank=item.ann_rank,
            bm25_rank=item.bm25_rank,
            ann_distance=item.chunk.ann_distance,
            bm25_score=item.chunk.bm25_score,
            rrf_score=item.rrf_score,
            reranker_score=None,
            mmr_score=None,
        ),
    )
    for position, item in enumerate(fused[:2], start=1)
)
print("QUERY")
print(f"  {query}")
print("\nFINAL RETRIEVAL EVIDENCE")
for rank, item in enumerate(evidence, start=1):
    print(f"  Rank {rank} | Source: {item.citation.source_name}")
    print(f"    Location: lines {item.citation.start_line}-{item.citation.end_line}")
    print(f"    Evidence: {item.content}")
print("\nRESULT")
print(f"  Returned {len(evidence)} citable evidence passages for generation.")

### What to observe

Expect two complete text fragments, source metadata, and ranking traces. Generation receives this
evidence—not a score-only summary.

### Conclusion

Retrieval is complete only when ranking ends in faithful, citable content.

## Optional appendix — live retrieval

Use `raglab-retrieve` after indexing a collection to explore PostgreSQL ANN/BM25, BGE reranking,
query rewriting, small-to-big expansion, filters, and MMR. Those service-dependent experiments
are intentionally outside this short lab.
